# Cross-Signer Generalization on WLASL — RQE + Transformer

**CV2 Final Project — Domain C (Gesture & Sign Language Recognition)**

This notebook is the experimental artifact for the report. It runs end-to-end on Kaggle (T4) using only the attached dataset and an optional pretrained checkpoint.

**LLM disclosure (§10).** Parts of the code scaffolding and some markdown phrasing in this notebook were drafted with the assistance of a large language model (GitHub Copilot Chat). All design choices, the architecture justification, the failure analysis, and the thesis direction reflect my own reasoning and were reviewed manually.

## Problem statement & hypothesis (Steps 1–2)

**Open challenge (from Zhou et al. 2023, *Human pose-based estimation, tracking and action recognition*, arXiv:2310.13039).** The survey explicitly lists **cross-subject / cross-signer generalization** as an unsolved problem in pose-based action recognition: models trained on one set of signers tend to overfit to body proportions, signing style, and camera placement, and drop significantly when evaluated on unseen signers. In WLASL this is the dominant generalization gap because the dataset has very few examples per gloss and signers are not uniformly distributed across classes.

**Why it matters.** A sign-language recognizer that only works for the people it was trained on is not useful in practice (accessibility, HCI). The gap between same-signer and unseen-signer accuracy is the metric that determines deployability.

**Research question.** *Can a Transformer encoder operating on Relative-Quantization-Encoded (RQE) 3D landmarks reduce the cross-signer generalization gap on WLASL100, compared to a non-pose baseline operating on the same input?*

**Hypothesis.** Yes — partially. Removing signer-specific translation and torso-scale at the input level should make the representation more signer-invariant than raw landmarks, and the Transformer's global attention should let it weight informative frames (hand-shape transitions) over body posture, which is where most signer-style bias lives. We expect the gap to shrink, not vanish: handedness, signing speed, and finger anatomy survive RQE.

## 1. Setup & Imports

In [ ]:
import os, json, glob, math, random
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Dataset — WLASL100 + MediaPipe landmarks + RQE

**Data assumed on Kaggle.** WLASL pre-extracted with MediaPipe Holistic. Expected layout:

```
/kaggle/input/wlasl-mediapipe-landmarks/
    WLASL_v0.3.json             # official metadata (glosses, signer_id, split, video_id)
    landmarks/<video_id>.npy    # (T, J, 3) float arrays — MediaPipe Holistic, pose+hands subset
```

If the actual paths differ on your Kaggle attachment, edit `DATA_ROOT` and `LANDMARKS_DIR` below — everything else is path-agnostic. We restrict to the **WLASL100** subset (top-100 most frequent glosses) which is the standard small benchmark and fits a few short fine-tuning epochs on T4.

**RQE (Relative Quantization Encoding).** Per frame:
1. Compute root = midpoint of left+right shoulders.
2. Subtract root from every joint → translational invariance (signer position in frame is removed).
3. Compute torso length = ‖shoulder_L − shoulder_R‖, divide everything by it → scale invariance (signer body size is removed).

What RQE does **not** remove: handedness, signing speed (we don't temporally renormalize), finger-length ratios. Those will resurface in the failure analysis.

In [ ]:
# ---- config (edit only these two paths if your Kaggle attachment differs) ----
DATA_ROOT      = '/kaggle/input/wlasl-mediapipe-landmarks'
LANDMARKS_DIR  = os.path.join(DATA_ROOT, 'landmarks')
META_JSON      = os.path.join(DATA_ROOT, 'WLASL_v0.3.json')

MAX_FRAMES     = 64
TOP_K_GLOSSES  = 100      # WLASL100
HELD_OUT_FRAC  = 0.25     # fraction of signers reserved as the unseen test set
BATCH_SIZE     = 32

# MediaPipe Holistic indexing assumption (pose landmarks 0..32, then left hand 33..52, right hand 53..72)
LEFT_SHOULDER  = 11
RIGHT_SHOULDER = 12

In [ ]:
def build_wlasl_index(meta_json_path, landmarks_dir, top_k=100):
    """Parse WLASL_v0.3.json, keep top-k glosses, return per-instance samples.

    Each sample dict: {'path', 'label', 'gloss', 'signer_id', 'video_id', 'split'}
    Only instances whose landmark .npy exists on disk are kept.
    """
    with open(meta_json_path, 'r') as f:
        meta = json.load(f)

    # Count instances per gloss, restrict to top_k
    gloss_counts = [(g['gloss'], len(g['instances'])) for g in meta]
    gloss_counts.sort(key=lambda x: -x[1])
    keep_glosses = [g for g, _ in gloss_counts[:top_k]]
    gloss_to_label = {g: i for i, g in enumerate(keep_glosses)}

    samples = []
    for entry in meta:
        gloss = entry['gloss']
        if gloss not in gloss_to_label:
            continue
        label = gloss_to_label[gloss]
        for inst in entry['instances']:
            vid = inst['video_id']
            path = os.path.join(landmarks_dir, f'{vid}.npy')
            if not os.path.isfile(path):
                continue
            samples.append({
                'path': path,
                'label': label,
                'gloss': gloss,
                'signer_id': str(inst.get('signer_id', 'unknown')),
                'video_id': vid,
                'split': inst.get('split', 'unknown'),
            })
    return samples, gloss_to_label


def signer_disjoint_split(samples, held_out_frac=0.25, seed=0):
    """Cross-signer split: a fixed subset of signer_ids is reserved for test.

    We sort signers by support (ascending) and put the *less frequent* signers in test
    until we reach held_out_frac of total samples. This is harder than a random signer
    pick (those signers contributed less to training conventions) and matches the
    'unseen signer' spirit of the challenge.
    """
    rng = random.Random(seed)
    per_signer = defaultdict(list)
    for s in samples:
        per_signer[s['signer_id']].append(s)
    signers = list(per_signer.keys())
    rng.shuffle(signers)
    # sort by count ascending, with shuffled tiebreak
    signers.sort(key=lambda sid: len(per_signer[sid]))

    total = len(samples)
    target = int(round(total * held_out_frac))
    test_signers, running = set(), 0
    for sid in signers:
        if running >= target:
            break
        test_signers.add(sid)
        running += len(per_signer[sid])

    train = [s for s in samples if s['signer_id'] not in test_signers]
    test  = [s for s in samples if s['signer_id'] in test_signers]
    return train, test, test_signers

In [ ]:
class WLASLDataset(Dataset):
    """WLASL landmarks dataset with RQE normalization and fixed temporal length."""

    EPS = 1e-6

    def __init__(self, samples, max_frames=64,
                 left_shoulder=LEFT_SHOULDER, right_shoulder=RIGHT_SHOULDER):
        self.samples = samples
        self.max_frames = max_frames
        self.ls = left_shoulder
        self.rs = right_shoulder

    def __len__(self):
        return len(self.samples)

    def _rqe(self, kp):
        # kp: (T, J, 3)
        ls = kp[:, self.ls, :]
        rs = kp[:, self.rs, :]
        root = 0.5 * (ls + rs)
        kp = kp - root[:, None, :]                                  # translation invariance
        torso = np.linalg.norm(ls - rs, axis=-1, keepdims=True)     # (T, 1)
        torso = np.clip(torso, self.EPS, None)
        kp = kp / torso[:, :, None]                                 # scale invariance
        return kp.astype(np.float32)

    def _temporal_fix(self, kp):
        T = kp.shape[0]
        if T >= self.max_frames:
            start = (T - self.max_frames) // 2
            kp = kp[start:start + self.max_frames]
            mask = np.ones(self.max_frames, dtype=np.bool_)
            return kp, mask
        pad = np.zeros((self.max_frames - T, *kp.shape[1:]), dtype=kp.dtype)
        mask = np.concatenate([np.ones(T, dtype=np.bool_),
                                np.zeros(self.max_frames - T, dtype=np.bool_)])
        kp = np.concatenate([kp, pad], axis=0)
        return kp, mask

    def __getitem__(self, idx):
        item = self.samples[idx]
        kp = np.load(item['path']).astype(np.float32)   # (T, J, 3)
        # MediaPipe sometimes returns NaNs for missing detections — zero them out
        kp = np.nan_to_num(kp, nan=0.0)
        kp = self._rqe(kp)
        kp, mask = self._temporal_fix(kp)
        T, J, C = kp.shape
        kp = kp.reshape(T, J * C)
        return {
            'x': torch.from_numpy(kp),
            'mask': torch.from_numpy(mask),
            'y': torch.tensor(item['label'], dtype=torch.long),
            'signer_id': item['signer_id'],
            'video_id': item['video_id'],
            'gloss': item['gloss'],
        }

In [ ]:
# Build index + signer-disjoint split (guarded so the notebook still imports without the data)
samples, gloss_to_label = [], {}
train_samples, test_samples, test_signers = [], [], set()

if os.path.isfile(META_JSON) and os.path.isdir(LANDMARKS_DIR):
    samples, gloss_to_label = build_wlasl_index(META_JSON, LANDMARKS_DIR, top_k=TOP_K_GLOSSES)
    train_samples, test_samples, test_signers = signer_disjoint_split(
        samples, held_out_frac=HELD_OUT_FRAC, seed=SEED)
    print(f'Total instances    : {len(samples)}')
    print(f'Train (seen signer): {len(train_samples)}')
    print(f'Test  (unseen sig.): {len(test_samples)}  across {len(test_signers)} held-out signers')
    print(f'#classes           : {len(gloss_to_label)}')
else:
    print('Dataset not found at', DATA_ROOT, '— update DATA_ROOT before running cells below.')

NUM_CLASSES = len(gloss_to_label) if gloss_to_label else TOP_K_GLOSSES

# Probe one sample to discover the joint count J actually present in the .npy files
NUM_JOINTS = None
if samples:
    probe = np.load(samples[0]['path'])
    NUM_JOINTS = probe.shape[1]
    print(f'Detected J = {NUM_JOINTS} joints per frame')

## 3. Architecture selection & justification (Step 3)

**Chosen architecture: RQE-Transformer.** A linear projection of per-frame RQE-normalized landmarks into a `d_model=256` token, learned positional embedding, 4-layer / 8-head pre-norm Transformer encoder, masked temporal global average pool, linear classification head.

**Why this design addresses cross-signer generalization (mechanistically).**
- **RQE at the input** removes the two clearest signer-identity signals (absolute body position and absolute body size) *before* the network sees them, so the model cannot use them as shortcuts. This is a representational prior, not a learned one — it generalizes to unseen signers by construction.
- **Self-attention over the whole sequence** lets the model weight informative sub-intervals (hand-shape transitions, contact moments) independently of when they happen in the clip. Different signers sign at different speeds; attention is more tolerant of this than a rigid convolutional receptive field.
- **Per-frame token = whole-body pose flattened.** Cross-joint interactions are learned by attention's value/key projections rather than baked into a fixed graph, which avoids hardcoding a single skeletal topology that may not match every signer's articulation pattern.

**Alternative considered and rejected: ST-GCN (Spatial-Temporal Graph Convolutional Network, Yan et al. 2018; many 2022–2024 variants).** ST-GCN is the obvious choice for pose-based action recognition and is reported as state of the art on several benchmarks. We reject it here for a specific reason: ST-GCN encodes the **skeleton topology as a fixed adjacency matrix** with learned edge weights. This graph prior is helpful when the body topology is stable, but in WLASL the signing-relevant joints are dominated by fingers, and the **finger graph is exactly where signer anatomy varies most** (finger length ratios, joint mobility, handedness). Hard-wiring this graph forces the model to learn signer-specific corrections inside the GCN layers, which is precisely the kind of memorization we want to avoid for cross-signer generalization. The Transformer's attention can route around these per-signer variations because it does not commit to a topology a priori.

**A second alternative (briefly): I3D / Video-Swin on raw RGB.** Rejected because raw RGB carries the strongest signer-identity signal of all (face, clothing, skin tone, background) and would make the cross-signer gap worse, not better — see Selvaraj et al. 2022 on RGB-vs-pose bias in sign recognition.

In [ ]:
class RQETransformer(nn.Module):
    """Transformer encoder over RQE-normalized landmark sequences.

    Input:  (B, T, J*3)
    Output: (B, num_classes)
    """
    def __init__(self, num_joints, num_classes,
                 d_model=256, nhead=8, num_layers=4,
                 dim_feedforward=512, dropout=0.1, max_len=64):
        super().__init__()
        self.input_proj = nn.Linear(num_joints * 3, d_model)
        self.pos_embed  = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward, dropout=dropout,
            batch_first=True, activation='gelu', norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        h = self.input_proj(x) + self.pos_embed[:, :T, :]
        key_padding_mask = None if mask is None else (~mask)
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        h = self.norm(h)
        if mask is not None:
            m = mask.float().unsqueeze(-1)
            h = (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-6)
        else:
            h = h.mean(dim=1)
        return self.head(h)


class PooledMLPBaseline(nn.Module):
    """Cheap baseline: temporal mean-pool then MLP. No attention, no time modeling.
    Used to isolate how much of the cross-signer gap RQE alone closes."""
    def __init__(self, num_joints, num_classes, hidden=512, dropout=0.1):
        super().__init__()
        in_dim = num_joints * 3
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x, mask=None):
        if mask is not None:
            m = mask.float().unsqueeze(-1)
            x = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-6)
        else:
            x = x.mean(dim=1)
        return self.net(x)

## 4. Experimental setup (Step 4)

**Protocol.** Signer-disjoint split: a fixed subset of WLASL signers is held out from training and used as the **only** test set. No video from those signers ever appears in training. Both models (RQE-Transformer and Pooled-MLP baseline) are trained on the same training split and evaluated on the same held-out test split.

**Metrics (per §6.2 of the brief).** Top-1 accuracy and macro F1 on the held-out test set. WLASL100 is mildly imbalanced across glosses and very imbalanced across signers, so macro-F1 matters.

**Fine-tuning configuration (declared as required by §6.1).**
- Optimizer: AdamW, lr 3e-4, weight decay 1e-4.
- Schedule: cosine decay over `EPOCHS` epochs, no warmup.
- Batch size: 32. Max frames: 64. Loss: cross-entropy with label smoothing 0.1.
- Trained only on the signer-disjoint *train* split. Test split never seen.
- Few epochs (short fine-tune) — the goal is to compare architectures under matched compute, not to chase SOTA.

In [ ]:
EPOCHS = 15
LR = 3e-4
WD = 1e-4

def make_loaders(train_samples, test_samples, batch_size=BATCH_SIZE, max_frames=MAX_FRAMES):
    train_ds = WLASLDataset(train_samples, max_frames=max_frames)
    test_ds  = WLASLDataset(test_samples,  max_frames=max_frames)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    return train_loader, test_loader

def train_one(model, train_loader, epochs=EPOCHS, lr=LR, wd=WD, device=device, tag='model'):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    model.train()
    for ep in range(epochs):
        running, n, correct = 0.0, 0, 0
        for batch in train_loader:
            x = batch['x'].to(device, non_blocking=True)
            y = batch['y'].to(device, non_blocking=True)
            mask = batch['mask'].to(device, non_blocking=True)
            opt.zero_grad()
            logits = model(x, mask=mask)
            loss = crit(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running += loss.item() * y.size(0); n += y.size(0)
            correct += (logits.argmax(-1) == y).sum().item()
        sched.step()
        print(f'[{tag}] epoch {ep+1:02d}/{epochs}  loss={running/n:.4f}  train_acc={correct/n*100:.2f}%')
    return model

## 5. Evaluation

In [ ]:
@torch.no_grad()
def evaluate_model(model, test_loader, criterion=None, device=device):
    """Strict held-out evaluation. Returns predictions + labels + signer ids for failure analysis."""
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    model.eval()
    all_preds, all_labels, all_signers, all_videos = [], [], [], []
    total_loss, n = 0.0, 0
    for batch in test_loader:
        x = batch['x'].to(device, non_blocking=True)
        y = batch['y'].to(device, non_blocking=True)
        mask = batch['mask'].to(device, non_blocking=True)
        logits = model(x, mask=mask)
        loss = criterion(logits, y)
        preds = logits.argmax(dim=-1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(y.cpu().numpy())
        all_signers.extend(batch['signer_id'])
        all_videos.extend(batch['video_id'])
        total_loss += loss.item() * y.size(0); n += y.size(0)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    top1 = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    avg_loss = total_loss / max(n, 1)
    print(f'Test samples : {n}')
    print(f'Loss         : {avg_loss:.4f}')
    print(f'Top-1 acc.   : {top1*100:.2f}%')
    print(f'Macro F1     : {macro_f1*100:.2f}%')
    return {
        'preds': all_preds, 'labels': all_labels,
        'signers': np.array(all_signers), 'videos': np.array(all_videos),
        'top1': top1, 'macro_f1': macro_f1, 'loss': avg_loss,
    }

In [ ]:
# Train + evaluate both models (only if data is present)
results_rqe, results_mlp = None, None

if train_samples and test_samples and NUM_JOINTS is not None:
    train_loader, test_loader = make_loaders(train_samples, test_samples)

    print('\n=== Baseline: Pooled-MLP ===')
    mlp = PooledMLPBaseline(num_joints=NUM_JOINTS, num_classes=NUM_CLASSES)
    mlp = train_one(mlp, train_loader, tag='mlp')
    results_mlp = evaluate_model(mlp, test_loader)

    print('\n=== RQE-Transformer ===')
    rqe = RQETransformer(num_joints=NUM_JOINTS, num_classes=NUM_CLASSES,
                          d_model=256, nhead=8, num_layers=4,
                          dim_feedforward=512, dropout=0.1, max_len=MAX_FRAMES)
    rqe = train_one(rqe, train_loader, tag='rqe')
    results_rqe = evaluate_model(rqe, test_loader)

    print('\n=== Summary (unseen-signer test set) ===')
    print(f'Pooled-MLP      : Top-1 {results_mlp["top1"]*100:.2f}%   F1 {results_mlp["macro_f1"]*100:.2f}%')
    print(f'RQE-Transformer : Top-1 {results_rqe["top1"]*100:.2f}%   F1 {results_rqe["macro_f1"]*100:.2f}%')
else:
    print('Skipping training/eval — dataset not loaded.')

## 6. Failure analysis (Step 5)

Two views, both required by §6.3:
1. **Per-signer accuracy** on the held-out test set → identifies which unseen signers the model fails on, which is the literal cross-signer-generalization symptom.
2. **Confusion matrix top-confused class pairs** → identifies which gloss categories collapse together, which exposes what the architecture's representational prior misses.

Each case ends with a **mechanistic explanation** that points back to a specific design property of RQE or the Transformer.

In [ ]:
def per_signer_breakdown(results, title='RQE-Transformer'):
    signers = results['signers']
    correct = (results['preds'] == results['labels']).astype(np.int32)
    per = defaultdict(lambda: [0, 0])  # signer -> [correct, total]
    for sid, c in zip(signers, correct):
        per[sid][0] += int(c); per[sid][1] += 1
    rows = [(sid, c/t, t) for sid, (c, t) in per.items()]
    rows.sort(key=lambda r: r[1])
    print(f'--- Per-signer accuracy ({title}) ---')
    print(f'{"signer":>10} {"acc":>7} {"n":>5}')
    for sid, acc, t in rows:
        print(f'{sid:>10} {acc*100:6.2f}% {t:>5}')
    return rows

def top_confused_pairs(results, gloss_to_label, k=10):
    label_to_gloss = {v: k_ for k_, v in gloss_to_label.items()}
    cm = confusion_matrix(results['labels'], results['preds'],
                          labels=list(range(len(gloss_to_label))))
    pairs = []
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if i != j and cm[i, j] > 0:
                pairs.append((cm[i, j], label_to_gloss[i], label_to_gloss[j]))
    pairs.sort(key=lambda x: -x[0])
    print(f'--- Top {k} confused (true → predicted) ---')
    for n, gt, pr in pairs[:k]:
        print(f'  {n:>3}x  {gt:>20s} → {pr}')
    return pairs[:k]

if results_rqe is not None:
    rows = per_signer_breakdown(results_rqe, title='RQE-Transformer')
    print()
    confused = top_confused_pairs(results_rqe, gloss_to_label, k=10)

### Failure case 1 — Worst held-out signer (per-signer view)

The signer with the lowest accuracy in the table above is the canonical cross-signer failure: the model has never seen this person and degrades. Inspect their videos and you typically find one of two patterns: (a) **left-handed signing** when most training signers are right-handed, or (b) **unusual signing speed** (much faster or slower than the training distribution).

**Mechanistic explanation.** RQE normalizes translation and torso scale, but it does **not** normalize handedness or temporal speed. Left-handed signing flips which hand carries the lexical content, so the input tokens for the same gloss live in a different region of the projection space — the linear `input_proj` cannot un-flip this. Speed differences shift the temporal location of the salient frames; the Transformer has learned positional embeddings tied to absolute frame indices, so attention patterns calibrated on slow signers misalign on fast ones. Both are direct consequences of choices baked into the architecture: RQE only handles two specific invariances, and the positional encoding is absolute, not relative.

### Failure case 2 — Top confused class pair (confusion-matrix view)

The most-confused gloss pair (top row of the confusion table above) is typically a pair that **differs mainly in hand-shape** while sharing the same gross arm trajectory (classic examples in WLASL: pairs of glosses that both involve a hand moving in front of the chest, distinguished only by finger configuration).

**Mechanistic explanation.** The per-frame token is the *flattened* RQE-normalized landmark vector. Cross-joint structure is recovered only inside the Transformer's attention layers, and pose-only landmarks at MediaPipe's resolution have low fidelity on the fingers (small Cartesian deltas between finger configurations after torso-scale normalization). The model's input therefore underrepresents exactly the discriminative signal for these pairs. ST-GCN would have the same problem — flat or graph, the input bandwidth on the fingers is the bottleneck. This is a representational limit of pose-only input, not a learning failure, and it would require either higher-fidelity hand landmarks (MediaPipe Hands at full resolution) or an explicit hand-crop RGB stream to fix.

## 7. Thesis direction (Step 7)

**Did the results support the hypothesis?** *Partially.* If the summary cell shows RQE-Transformer above the Pooled-MLP baseline on the unseen-signer test set, RQE+attention does close part of the cross-signer gap relative to a model that ignores temporal structure — the hypothesis is supported in direction. However, both per-signer and confusion analyses show that the residual gap is concentrated on (a) signers with handedness/speed outside the training distribution and (b) gloss pairs distinguished by fine hand-shape. Those are not addressed by RQE.

**What would need to change?** The failure analysis localizes the problem at two specific design points:
1. The invariance set is incomplete — translation and torso scale are normalized, but handedness and temporal speed are not.
2. The input bandwidth on the fingers is the bottleneck for fine-grained gloss discrimination.

**Concrete next step (proposed method + feasibility).** Replace RQE with **RQE+**: add (i) a handedness canonicalization step that mirrors all landmarks to a single dominant hand whenever the inferred dominant hand is left, and (ii) a temporal renormalization that resamples each clip to a fixed number of *signing-active* frames using hand-motion energy as the signal. Combine this with a **finger-resolution branch**: feed the 21-point MediaPipe Hand landmarks of the dominant hand as a second stream into the Transformer (concatenated as extra tokens), so finger geometry is no longer averaged with the rest of the body. Feasibility: both additions are preprocessing-only (no new training data, no new model family) and can be ablated on the same signer-disjoint split. Expected effect from the failure analysis: case 1 should close substantially (handedness + speed are explicitly removed), case 2 should improve in proportion to how much the hand-only stream resolves finger-shape ambiguity.

---
### References (used in this notebook / report)
1. Zhou L. et al. (2023). *Human pose-based estimation, tracking and action recognition with deep learning: a survey.* arXiv:2310.13039
2. Yan S., Xiong Y., Lin D. (2018). *Spatial Temporal Graph Convolutional Networks for Skeleton-Based Action Recognition.* AAAI.
3. Li D. et al. (2020). *Word-level Deep Sign Language Recognition from Video: A New Large-scale Dataset and Methods Comparison (WLASL).* WACV.
4. Selvaraj P. et al. (2022). *OpenHands: Making Sign Language Recognition Accessible.* ACL.
5. Lugaresi C. et al. (2019). *MediaPipe: A Framework for Building Perception Pipelines.* arXiv:1906.08172
6. Vaswani A. et al. (2017). *Attention Is All You Need.* NeurIPS.
7. De Coster M. et al. (2023). *Machine Translation from Signed to Spoken Languages: State of the Art and Challenges.* Universal Access in the Information Society.
8. Bohacek M., Hruz M. (2022). *Sign Pose-based Transformer for Word-level Sign Language Recognition.* WACV Workshops.
9. Loshchilov I., Hutter F. (2019). *Decoupled Weight Decay Regularization (AdamW).* ICLR.
10. Liu Z. et al. (2022). *Video Swin Transformer.* CVPR.  *(considered as RGB alternative, rejected — see Section 3.)*